## 24. Merton Structural Model — Public Company Credit Risk

A different lens entirely: instead of modeling consumer loans statistically, the Merton model treats a public company's equity as a call option on its assets, and backs out an implied Distance to Default and PD from market-observable equity price and volatility plus balance-sheet debt. Applied here to a mix of investment-grade and high-yield names to sanity-check whether the model's implied risk ranking lines up with conventional credit-quality groupings.

In [ ]:
# ============================================================
# CELL 1 — Pull stock price + balance sheet data via yfinance
# ============================================================
import numpy as np
import pandas as pd
!pip install yfinance --quiet
import yfinance as yf

# Mix of investment grade and high yield, per the roadmap
tickers = [
    'AAPL', 'MSFT', 'JNJ', 'PG', 'KO',          # investment grade, stable
    'WMT', 'HD', 'V', 'UNH', 'COST',            # investment grade
    'F', 'CCL', 'AAL', 'X', 'OXY',              # high yield / more leveraged
    'DAL', 'UAL', 'RCL', 'NCLH', 'MGM'          # high yield, cyclical
]

company_data = {}
for ticker in tickers:
    try:
        tk = yf.Ticker(ticker)
        info = tk.info
        hist = tk.history(period='1y')

        company_data[ticker] = {
            'equity_market_value': info.get('marketCap', np.nan),
            'total_debt': info.get('totalDebt', np.nan),
            'shares_outstanding': info.get('sharesOutstanding', np.nan),
            'stock_price_history': hist['Close'],
            'equity_volatility_annualized': hist['Close'].pct_change().std() * np.sqrt(252),
            'sector': info.get('sector', 'Unknown'),
        }
        print(f"{ticker}: market cap=${info.get('marketCap', 0):,.0f}, "
              f"debt=${info.get('totalDebt', 0):,.0f}, "
              f"equity vol={company_data[ticker]['equity_volatility_annualized']:.3f}")
    except Exception as e:
        print(f"{ticker}: FAILED -- {e}")

print(f"\nSuccessfully pulled data for {len(company_data)} / {len(tickers)} companies")

In [ ]:
# ============================================================
# CELL 2 — Build a clean dataframe of Merton model inputs
# ============================================================
merton_input_rows = []
for ticker, data in company_data.items():
    if pd.isna(data['equity_market_value']) or pd.isna(data['total_debt']):
        continue
    merton_input_rows.append({
        'ticker': ticker,
        'E': data['equity_market_value'],            # market value of equity
        'D': data['total_debt'],                      # debt face value (strike)
        'sigma_E': data['equity_volatility_annualized'],  # observed equity volatility
        'sector': data['sector'],
    })

merton_df = pd.DataFrame(merton_input_rows).dropna()
merton_df = merton_df[merton_df['D'] > 0]  # need positive debt for the model to be meaningful

print(f"Companies with usable Merton inputs: {len(merton_df)}")
print(merton_df[['ticker', 'E', 'D', 'sigma_E']].to_string(index=False))

In [ ]:
# ============================================================
# CELL 3 — Solve the Merton model: iteratively find V and sigma_V
# ============================================================
from scipy.stats import norm
from scipy.optimize import brentq

RISK_FREE_RATE = 0.045   # approx 1-year T-bill rate, reasonable current proxy
TIME_HORIZON = 1.0       # 1-year horizon, standard Merton convention

def merton_equations(V, sigma_V, E, D, r, T):
    """
    Returns (model-implied E, model-implied sigma_E*E) given a guess of (V, sigma_V).
    These should match the OBSERVED E and sigma_E*E when the guess is correct.
    """
    if V <= 0 or sigma_V <= 0:
        return np.inf, np.inf
    d1 = (np.log(V / D) + (r + 0.5 * sigma_V**2) * T) / (sigma_V * np.sqrt(T))
    d2 = d1 - sigma_V * np.sqrt(T)
    E_model = V * norm.cdf(d1) - D * np.exp(-r * T) * norm.cdf(d2)
    sigma_E_times_E_model = norm.cdf(d1) * sigma_V * V
    return E_model, sigma_E_times_E_model

def solve_merton(E_obs, sigma_E_obs, D, r=RISK_FREE_RATE, T=TIME_HORIZON,
                  max_iter=200, tol=1e-6):
    """
    Iteratively solve for V and sigma_V such that the Merton equations
    reproduce the observed equity value and equity volatility.
    """
    # Initial guesses: V starts at E + D (rough), sigma_V starts at sigma_E
    V = E_obs + D
    sigma_V = sigma_E_obs

    for iteration in range(max_iter):
        # Step 1: holding sigma_V fixed, solve for V such that E_model == E_obs
        def equity_gap(V_guess):
            E_model, _ = merton_equations(V_guess, sigma_V, E_obs, D, r, T)
            return E_model - E_obs

        try:
            V_new = brentq(equity_gap, E_obs * 0.5, (E_obs + D) * 3, xtol=1e-4)
        except ValueError:
            V_new = V  # bracket failed, keep previous guess this iteration

        # Step 2: holding V fixed, recompute sigma_V from the volatility equation
        d1 = (np.log(V_new / D) + (r + 0.5 * sigma_V**2) * T) / (sigma_V * np.sqrt(T))
        sigma_V_new = sigma_E_obs * E_obs / (norm.cdf(d1) * V_new)

        # Check convergence
        if abs(V_new - V) < tol * V and abs(sigma_V_new - sigma_V) < tol * sigma_V:
            V, sigma_V = V_new, sigma_V_new
            break

        V, sigma_V = V_new, sigma_V_new

    return V, sigma_V, iteration + 1

print("Solving Merton model for each company...")
merton_results = []
for _, row in merton_df.iterrows():
    V, sigma_V, n_iter = solve_merton(row['E'], row['sigma_E'], row['D'])
    merton_results.append({
        'ticker': row['ticker'],
        'E': row['E'], 'D': row['D'], 'sigma_E': row['sigma_E'],
        'V': V, 'sigma_V': sigma_V, 'n_iter': n_iter
    })

merton_solved_df = pd.DataFrame(merton_results)
print(merton_solved_df[['ticker', 'E', 'D', 'V', 'sigma_E', 'sigma_V', 'n_iter']].to_string(index=False))

In [ ]:
# ============================================================
# CELL 4 — Compute Distance to Default and Merton PD
# ============================================================

# Distance to Default: how many standard deviations of asset value sit
# between the firm's current asset value and its default threshold (debt).
# A higher DD means more cushion before the firm would default --> safer.
merton_solved_df['distance_to_default'] = (
    (merton_solved_df['V'] - merton_solved_df['D']) /
    (merton_solved_df['V'] * merton_solved_df['sigma_V'])
)

# Merton PD: probability that a standard normal draw falls below -DD,
# i.e. the probability that asset value drops enough to breach the
# default threshold within the model's time horizon.
merton_solved_df['merton_pd'] = norm.cdf(-merton_solved_df['distance_to_default'])

# Sort by risk, most to least risky
merton_solved_df = merton_solved_df.sort_values('merton_pd', ascending=False)

print("--- Merton Model Results: Distance to Default and Implied PD ---")
print(merton_solved_df[['ticker', 'V', 'D', 'sigma_V', 'distance_to_default', 'merton_pd']]
      .to_string(index=False, formatters={
          'V': '{:,.0f}'.format,
          'D': '{:,.0f}'.format,
          'sigma_V': '{:.4f}'.format,
          'distance_to_default': '{:.3f}'.format,
          'merton_pd': '{:.4%}'.format,
      }))

In [ ]:
# ============================================================
# CELL 5 — Visualize: rank companies by Merton-implied default risk
# ============================================================
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

plot_df = merton_solved_df.sort_values('distance_to_default')
colors = ['#e74c3c' if dd < 3 else '#f39c12' if dd < 6 else '#2ecc71'
          for dd in plot_df['distance_to_default']]

axes[0].barh(plot_df['ticker'], plot_df['distance_to_default'], color=colors)
axes[0].set_title('Distance to Default by Company\n(lower = riskier)')
axes[0].set_xlabel('Distance to Default (std. deviations)')
axes[0].axvline(3, color='gray', linestyle='--', alpha=0.5, label='DD = 3 (commonly cited risk threshold)')
axes[0].legend()

plot_df2 = merton_solved_df.sort_values('merton_pd')
axes[1].barh(plot_df2['ticker'], plot_df2['merton_pd'] * 100, color=colors[::-1])
axes[1].set_title('Merton-Implied 1-Year PD by Company')
axes[1].set_xlabel('Implied PD (%)')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 6 — Sanity check: does the Merton model's ranking make sense?
# ============================================================
# Cross-check against the rough investment-grade / high-yield split we
# deliberately built into the ticker list at the start.

investment_grade = ['AAPL', 'MSFT', 'JNJ', 'PG', 'KO', 'WMT', 'HD', 'V', 'UNH', 'COST']
high_yield = ['F', 'CCL', 'AAL', 'OXY', 'DAL', 'UAL', 'RCL', 'NCLH', 'MGM']

merton_solved_df['expected_category'] = merton_solved_df['ticker'].apply(
    lambda t: 'Investment Grade' if t in investment_grade else 'High Yield'
)

print("--- Mean Merton PD by Expected Credit Category ---")
print(merton_solved_df.groupby('expected_category')['merton_pd'].agg(['mean', 'min', 'max']).round(6))

print("\n--- Does the model's own ranking line up with the expected split? ---")
print(merton_solved_df[['ticker', 'expected_category', 'merton_pd']].to_string(index=False))